In [1]:
import numpy as np

from numba import njit

from scipy.integrate import solve_ivp

from scipy.linalg import eigh

from sklearn.linear_model import Ridge, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, root_mean_squared_error, accuracy_score

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

import fastplotlib as fpl

from dysts.maps import Henon

Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x01,\x00\x00\x007\x08\x06\x00\x00\x00\xb6\x1bw\x99\x…

Valid,Device,Type,Backend,Driver
✅ (default),Apple M4,IntegratedGPU,Metal,


To silence this warning, use a fully namespaced name.


# Init

In [2]:
steps = 20000

tau_steps = 1

transient_steps_chaos = int(steps * 0.1)
transient_steps_reservoir = int(steps * 0.1)

total_steps = steps + transient_steps_chaos + transient_steps_reservoir + tau_steps
total_steps_after_chaos = steps + transient_steps_reservoir + tau_steps

test_size = 0.2
test_steps = int(steps * test_size)

t = np.arange(0, total_steps)

In [3]:
henon_model = Henon()
henon_dataset = henon_model.make_trajectory(total_steps)
henon_dataset = henon_dataset[transient_steps_chaos:]

In [4]:
henon_scaler = StandardScaler()
henon_train_scaled = henon_scaler.fit_transform(henon_dataset[:-test_steps])
henon_test_scaled = henon_scaler.transform(henon_dataset[-test_steps:])
henon_scaled = np.concatenate((henon_train_scaled, henon_test_scaled), axis=0)

In [5]:
def scatter_plot(data_list, names=["Test", "Pred"], colors=["white", "magenta"]):
    fig = go.Figure()

    for i, data in enumerate(data_list):
        if data.shape[1] >= 3:
            fig.add_trace(
                go.Scatter3d(
                    x=data[:, 0],
                    y=data[:, 1],
                    z=data[:, 2],
                    mode="markers",
                    name=names[i],
                    marker=dict(color=colors[i % len(colors)], size=1),
                )
            )
        else:
            fig.add_trace(
                go.Scattergl(
                    x=data[:, 0],
                    y=data[:, 1],
                    mode="markers",
                    name=names[i],
                    marker=dict(color=colors[i % len(colors)], size=1),
                )
            )

    fig.update_layout(template="plotly_dark", title="Attractor Comparison")

    return fig

In [6]:
def r_2_plots_grid(actual_list, predicted_list, titles):
    fig = make_subplots(rows=1, cols=actual_list.shape[1], subplot_titles=titles)

    for i in range(actual_list.shape[1]):
        actual = actual_list[:, i]
        predicted = predicted_list[:, i]
        r_2 = r2_score(actual, predicted)
        col = i + 1

        fig.add_trace(
            go.Scatter(
                x=actual,
                y=predicted,
                mode="markers",
                name="Data",
                marker=dict(color="rgba(50, 50, 200, 0.5)", size=5),
            ),
            row=1,
            col=col,
        )

        min_val, max_val = min(actual.min(), predicted.min()), max(
            actual.max(), predicted.max()
        )
        fig.add_trace(
            go.Scatter(
                x=[min_val, max_val],
                y=[min_val, max_val],
                mode="lines",
                name="Ideal",
                line=dict(color="firebrick", dash="dash"),
            ),
            row=1,
            col=col,
        )

        fig.update_xaxes(title_text="Actual", row=1, col=col)
        fig.update_yaxes(title_text=f"Predicted (R²: {r_2:.4f})", row=1, col=col)

    fig.update_layout(showlegend=False, height=500, width=1000)
    return fig

In [7]:
def plot_grid(nodes_pos):
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=nodes_pos[:, 0],
            y=nodes_pos[:, 1],
            mode="markers",
            marker=dict(size=8, color="teal"),
        )
    )

    fig.update_layout(
        title="Hexagonal Lattice Distribution",
        xaxis=dict(title="X Position", scaleanchor="y", scaleratio=1),
        yaxis=dict(title="Y Position"),
        plot_bgcolor="white",
        width=700,
        height=700,
    )

    return fig


In [8]:
def weight_plot(weights):
    labels = [
        f"{'Pos' if i % 2 == 0 else 'Vel'} Node {i//2 + 1}" for i in range(len(weights))
    ]

    fig = go.Figure(
        data=[
            go.Bar(
                x=labels,
                y=weights,
                marker_color=np.where(weights >= 0, "royalblue", "firebrick"),
            )
        ]
    )

    fig.update_layout(
        title="Reservoir Node Contribution (Feature Weights)",
        xaxis_title="Spring/Mass Node",
        yaxis_title="Weight Value",
        template="plotly_white",
    )

    return fig

In [9]:
def spring_animation(
    disp,
    nodes_pos,
    connections_list,
    size=15,
    external=False,
    is_3d=False,
    frames_moved=5,
    max_frames=2000,
    animate=True,
    highlighted_nodes=None,
    vel=None,
    steps_jump=1,
):
    disp = disp[::steps_jump]

    steps = disp.shape[0]
    num_nodes = nodes_pos.shape[0]
    dims = nodes_pos.shape[1]

    disp_reshaped = disp.reshape(steps, num_nodes, dims)
    disp_3d = np.pad(disp_reshaped, ((0, 0), (0, 0), (0, 3 - dims)), mode="constant")
    nodes_pos_3d = np.pad(nodes_pos, ((0, 0), (0, 3 - dims)), mode="constant")

    fig = (
        fpl.Figure(canvas="glfw" if external else "jupyter")
        if not is_3d
        else fpl.Figure(
            cameras="3d",
            controller_types="orbit",
            canvas="glfw" if external else "jupyter",
        )
    )

    node_colors = np.array(["magenta"] * num_nodes)
    if highlighted_nodes is not None:
        node_colors[highlighted_nodes] = "lime"

    coords = nodes_pos_3d + disp_3d[0]
    dots = fig[0, 0].add_scatter(
        data=coords.astype(np.float32), sizes=size, colors=node_colors
    )

    lines = [
        fig[0, 0].add_line(
            data=np.vstack([coords[int(row[0])], coords[int(row[1])]]).astype(
                np.float32
            ),
            thickness=2,
            colors="cyan",
        )
        for row in connections_list
    ]

    if vel is not None:
        vel = vel[::steps_jump]
        vel_reshaped = vel.reshape(steps, num_nodes, dims)
        vel_3d = np.pad(vel_reshaped, ((0, 0), (0, 0), (0, 3 - dims)), mode="constant")

        vel_lines = [
            fig[0, 0].add_line(
                data=np.vstack([coords[i], coords[i] + vel_3d[0, i]]).astype(np.float32),
                thickness=1.5,
                colors="yellow",
            )
            for i in range(num_nodes)
        ]

    frame_tracker = 0
    def update_springs(canvas):
        nonlocal frame_tracker
        frame_tracker = (frame_tracker + frames_moved) % steps
        if frame_tracker >= max_frames:
            canvas.clear_animations()

        coords = nodes_pos_3d + disp_3d[frame_tracker]

        dots.data = coords.astype(np.float32)

        for row, l in zip(connections_list, lines):
            src, dst = int(row[0]), int(row[1])
            l.data = np.vstack([coords[src], coords[dst]]).astype(np.float32)

        if vel is not None:
            for i in range(num_nodes):
                vel_lines[i].data = np.vstack(
                    [coords[i], coords[i] + vel_3d[frame_tracker, i]]
                ).astype(np.float32)

    if animate:
        fig.add_animations(update_springs)
    return fig

In [10]:
def plot_grid(nodes_pos):
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=nodes_pos[:, 0],
            y=nodes_pos[:, 1],
            mode="markers",
            marker=dict(size=8, color="teal"),
        )
    )

    fig.update_layout(
        title="Hexagonal Lattice Distribution",
        xaxis=dict(title="X Position", scaleanchor="y", scaleratio=1),
        yaxis=dict(title="Y Position"),
        plot_bgcolor="white",
        width=700,
        height=700,
    )

    return fig

# Calc Init

In [11]:
# @njit(fastmath=True, cache=True)
def get_spring_forces(connections_list, disp, initial_pos, rest_lens, k_vals, num_nodes, dims):
    forces = np.zeros((num_nodes, dims))

    idx_a, idx_b = connections_list[:, 0], connections_list[:, 1]

    disp_reshaped = disp.reshape(
        num_nodes, dims
    )  # So now for every index/time_step we have a 2D array where every index is x,y,z

    pos_a = initial_pos[idx_a] + disp_reshaped[idx_a]
    pos_b = initial_pos[idx_b] + disp_reshaped[idx_b]

    r_vecs = pos_b - pos_a
    current_lens = np.linalg.norm(r_vecs)

    force_magnitudes = k_vals * (current_lens - rest_lens)

    safe_lens = np.where(current_lens < 1e-13, 1.0, current_lens)
    unit_dirs = r_vecs / safe_lens.reshape(-1, 1)

    forces[idx_a] += force_magnitudes[:, np.newaxis] * unit_dirs
    forces[idx_b] -= force_magnitudes[:, np.newaxis] * unit_dirs

    return forces.reshape(-1)

In [28]:
# @njit(fastmath=True, cache=True)
def run_simulation(
    steps, dt, m_inv_diag, c_diag, U, initial_pos, connections_list, k_vals, wall_nodes=[-1]
):
    num_nodes = initial_pos.shape[0]
    dims = initial_pos.shape[1]
    matrix_size = num_nodes * dims

    disp = np.zeros((steps, matrix_size))
    v = np.zeros((steps, matrix_size))
    acc = np.zeros(matrix_size)

    init_vecs = (
        initial_pos[connections_list[:, 0]] - initial_pos[connections_list[:, 1]]
    )
    rest_lens = np.linalg.norm(init_vecs)

    mask = np.ones(matrix_size)
    if wall_nodes[0] != -1:
        for wall in wall_nodes:
            idx = wall * dims
            mask[idx : idx + dims] = 0

    F_spring = get_spring_forces(
        connections_list, disp[0], initial_pos, rest_lens, k_vals, num_nodes, dims
    )
    F_spring *= mask

    for i in range(1, steps):
        acc = m_inv_diag * (F_spring - c_diag * v[i - 1] + U[i - 1])

        disp[i] = disp[i - 1] + v[i - 1] * dt + acc * 0.5 * dt**2

        F_spring = get_spring_forces(
            connections_list, disp[i], initial_pos, rest_lens, k_vals, num_nodes, dims
        )
        F_spring *= mask

        acc_next = m_inv_diag * (F_spring - c_diag * (v[i - 1] + acc * dt) + U[i])

        v[i] = v[i - 1] + 0.5 * (acc + acc_next) * dt

    return disp, v

# Single Hex

In [13]:
side_len = 1
x = np.array(
    [-side_len / 2, side_len / 2, side_len, side_len / 2, -side_len / 2, -side_len]
)
y = np.array(
    [
        0,
        0,
        side_len * np.sqrt(3) / 2,
        side_len * np.sqrt(3),
        side_len * np.sqrt(3),
        side_len * np.sqrt(3) / 2,
    ]
)
nodes_pos = np.column_stack((x, y))

In [14]:
rng = np.random.default_rng(42)

tau_steps = 1

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

node_m = rng.uniform(0.1, 0.3, size=num_nodes)
m_inv_diag = 1.0 / np.repeat(node_m, dims)

node_c = rng.uniform(0.05, 0.3, size=num_nodes)
c_diag = np.repeat(node_c, dims)

node_ids = np.arange(x.size)
src_nodes = node_ids
dst_nodes = np.roll(node_ids, 1)
k_vals = np.ones(src_nodes.shape[0]) * 0.7
connections_list = np.column_stack((src_nodes, dst_nodes))

In [15]:
U = np.zeros((steps + transient_steps_reservoir + tau_steps, matrix_size))
target_nodes = np.array([2, 5])
col_indices = (target_nodes[:, None] * dims + np.arange(dims)).reshape(-1)
vectorized_force = np.zeros((U.shape[0], len(col_indices)))
# vectorized_force[:, 0] = henon_scaled[:, 0]
# vectorized_force[:, 2] = henon_scaled[:, 1]
vectorized_force[0, 0] = 100
U[:, col_indices] = vectorized_force

In [16]:
displacement, velocity = run_simulation(
    steps=steps + transient_steps_reservoir + tau_steps,
    dt=0.01,
    m_inv_diag=m_inv_diag,
    c_diag=c_diag,
    U=U,
    initial_pos=nodes_pos,
    connections_list=connections_list,
    k_vals=k_vals,
    wall_nodes=[0, 1, 3, 4]
)

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[ 0.          0.         -0.          0.         -0.00225622  0.
 -0.         -0.          0.         -0.          0.00217614  0.        ]
[ 0.          0.         -0.          0.         -0.00472229  0.
 -0.         -0.          0.         -0.          0.00439922  0.        ]


In [ ]:
spring_animation(
    disp=displacement,
    nodes_pos=nodes_pos,
    connections_list=connections_list,
    size=10,
    highlighted_nodes=target_nodes,
    external=True,
    max_frames=20000,
).show()

2026-07-15 11:19:10.581 python[92563:4234527] +[IMKClient subclass]: chose IMKClient_Modern
2026-07-15 11:19:10.581 python[92563:4234527] +[IMKInputSession subclass]: chose IMKInputSession_Modern


# Single Hex

In [18]:
side_len = 1
x = np.array(
    [side_len / 2, side_len, side_len / 2]
)
y = np.array(
    [
        0,
        side_len * np.sqrt(3) / 2,
        side_len * np.sqrt(3),
    ]
)
nodes_pos = np.column_stack((x, y))

In [19]:
rng = np.random.default_rng(42)

tau_steps = 1

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

node_m = rng.uniform(0.1, 0.3, size=num_nodes)
m_inv_diag = 1.0 / np.repeat(node_m, dims)

node_c = rng.uniform(0.05, 0.3, size=num_nodes)
c_diag = np.repeat(node_c, dims)

node_ids = np.arange(x.size)
src_nodes = node_ids
dst_nodes = np.roll(node_ids, 1)
k_vals = np.ones(src_nodes.shape[0]) * 0.7
connections_list = np.column_stack((src_nodes, dst_nodes))

In [35]:
U = np.zeros((steps + transient_steps_reservoir + tau_steps, matrix_size))
target_nodes = np.array([1])
col_indices = (target_nodes[:, None] * dims + np.arange(dims)).reshape(-1)
vectorized_force = np.zeros((U.shape[0], len(col_indices)))
# vectorized_force[:, 0] = henon_scaled[:, 0]
# vectorized_force[:, 2] = henon_scaled[:, 1]
vectorized_force[0, 0] = 1000
U[:, col_indices] = vectorized_force

In [36]:
displacement, velocity = run_simulation(
    steps=steps + transient_steps_reservoir + tau_steps,
    dt=0.01,
    m_inv_diag=m_inv_diag,
    c_diag=c_diag,
    U=U,
    initial_pos=nodes_pos,
    connections_list=connections_list,
    k_vals=k_vals,
    wall_nodes=[0, 2]
)

In [38]:
spring_animation(
    disp=displacement,
    nodes_pos=nodes_pos,
    connections_list=connections_list,
    size=10,
    highlighted_nodes=target_nodes,
    external=True,
    max_frames=20000,
).show()